In [ ]:
import numpy as np
import cupy as cp
from scipy import ndimage
from types import SimpleNamespace

from holotomocupy.rec import Rec
from holotomocupy.utils import *

# Acquisition parameters

In [ ]:
n = 128
ntheta = 360
detector_pixelsize = 1.4760147601476e-6 * 2 * 2048/n
energy = 17.1
wavelength = 1.24e-09 / energy
focustodetectordistance = 1.217

sx0 = -3.135e-3
z1 = np.array([5.110, 5.464, 6.879, 9.817, 10.372, 11.146, 12.594, 17.209]) * 1e-3 - sx0
z1_ids = np.array([0, 1, 2, 3]) ### note using index starting from 0
z1 = z1[z1_ids]
ndist = len(z1)
z2 = focustodetectordistance - z1
distances = (z1 * z2) / focustodetectordistance
magnifications = focustodetectordistance / z1
norm_magnifications = magnifications / magnifications[0]

nobj = int(np.ceil(n / norm_magnifications[-1] / 64)) * 64 
theta = cp.linspace(0,np.pi,ntheta).astype('float32')


# Reconstrution parameters

In [ ]:
rec_args = SimpleNamespace(
    n = n,
    nz = n,
    nobj = nobj,
    nzobj = nobj,
    detector_pixelsize = detector_pixelsize,      
    ndist=ndist,
    ntheta=ntheta,
    energy=energy,        
    focustodetectordistance=focustodetectordistance,        
    z1=z1,   
    theta=theta,
    ngpus=1,
    mask=1.1,   
    lam_prbfit=3e-3,
    lam_reg=0,
    obj_dtype='complex64',
    rho=[1,0.1,0.05],          
    nchunk = 16,
    niter = 513,
    vis_step = 32,
    err_step = 32,
    path_out = f"/local/ssd/tmp_holo",    
    start_iter = 0,
    show=True,            
)

# create class
cl = Rec(rec_args)

# Read data

In [ ]:
data=np.load(f'data/data_{n}.npy')
ref=np.load(f'data/ref_{n}.npy')

## Read intiial guess

In [ ]:
obj_init = np.load(f'rec/rec_init_{n}.npy')
prb_init= cp.ones([ndist,n,n],dtype='complex64')
pos_init= np.load(f'data/pos_{n}.npy')

# Iterative reconstruction

In [ ]:
vars = {"obj":obj_init, "prb":prb_init, "pos":pos_init}    
cl.BH(np.sqrt(data),cp.array(np.sqrt(ref)),vars)